# Análisis de sentimiento con NLP y Machine Learning
Notebook educativo basado en el dataset `data/comentarios.csv`.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay


## 1. Carga de datos

In [ ]:
df = pd.read_csv('../data/comentarios.csv')
df.head()


In [ ]:
print('Dimensiones:', df.shape)
print('\nDistribución de clases:')
print(df['tipo'].value_counts())
print('\nDuplicados por comentario:', df.duplicated(subset=['comentario']).sum())


## 2. Limpieza y normalizaciónSe preservan las negaciones porque expresiones como **no me gusta** son relevantes para el sentimiento.

In [ ]:
def limpiar_texto(texto):
    texto = str(texto).lower().strip()
    texto = re.sub(r'https?://\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'@\w+', ' ', texto)
    texto = re.sub(r'#(\w+)', r'\1', texto)
    texto = re.sub(r'\d+', ' ', texto)
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

df['comentario_limpio'] = df['comentario'].apply(limpiar_texto)
df[['comentario','comentario_limpio']].head()


## 3. Eliminación de duplicadosSe eliminan duplicados antes del split para reducir el riesgo de **data leakage**.

In [ ]:
df_modelo = df.drop_duplicates(subset=['comentario_limpio']).reset_index(drop=True)
print('Registros originales:', len(df))
print('Registros únicos:', len(df_modelo))
df_modelo['tipo'].value_counts()


## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_modelo['comentario_limpio'],
    df_modelo['tipo'],
    test_size=0.20,
    random_state=42,
    stratify=df_modelo['tipo']
)


## 5. TF-IDF + Regresión Logística

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, sublinear_tf=True)),
    ('modelo', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
pipeline.fit(X_train, y_train)


## 6. Evaluación

In [ ]:
pred = pipeline.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45)
plt.title('Matriz de confusión')
plt.tight_layout()
plt.show()


## 7. Predicciones sobre nuevos comentarios

In [ ]:
ejemplos = [
    'La camioneta está excelente, me encantó',
    'No me gustó para nada, fue una mala experiencia',
    'Quiero más información sobre el crédito'
]
for texto, prediccion in zip(ejemplos, pipeline.predict(ejemplos)):
    print(f'{texto} -> {prediccion}')


## ConclusiónEste baseline permite demostrar un pipeline completo de NLP clásico. El siguiente paso recomendado es comparar el rendimiento con Naive Bayes, SVM y posteriormente con embeddings o Transformers.